# gist: GPU batching experiment

Measures whether the worker's request batching pays off on a GPU. Run on a **T4 runtime** (Runtime → Change runtime type → T4 GPU), top to bottom, in one session so every run shares the same GPU, torch and Documents.

1. **Beams experiment** (model only): latency and ROUGE for beam counts 1, 2, 4, 6. Pick `BEAM_COUNT` from it.
2. **Batch-size sweep** (model only): throughput at batch sizes 1–64.
3. **End-to-end concurrency sweeps** against a worker on `localhost`: the no-queue baseline, then the queued worker at batch sizes 1, 8 and 32.

Fixed everywhere: fp32, `BATCH_WAIT_MS=50`, the same Documents in the same order. Uses Colab's preinstalled torch; every CSV row records the GPU, torch/CUDA versions and commit.

In [ ]:
!nvidia-smi

## Setup

Clones the experiment branch, plus the `baseline-no-queue` branch as a second checkout. Both must be pushed to GitHub first.

In [ ]:
%cd /content
!rm -rf gist gist-baseline
!git clone -q -b gpu-batching-experiment https://github.com/adriankalisz/gist.git
!git -C gist worktree add -q /content/gist-baseline origin/baseline-no-queue
!git -C gist log --oneline -1
!git -C gist-baseline log --oneline -1

In [ ]:
# Everything except torch, which stays Colab's preinstalled build
!pip install -q "fastapi>=0.115" "uvicorn[standard]>=0.32" "transformers>=4.45" "datasets>=3.0" "pydantic-settings>=2.15.0" "requests>=2.31" "rouge-score>=0.1.2"

In [ ]:
%cd /content/gist
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
RESULTS = "/content/gist/quick_tests/results"

## 1. Beams experiment

In [ ]:
!python quick_tests/beams_experiment.py --out {RESULTS}/beams_experiment.csv

Pick the beam count to use for everything below, trading ROUGE against latency.

In [ ]:
BEAM_COUNT = 2  # <- set from the table above

### Optional: check ROUGE against the published score

Our ROUGE-2 (~0.15) is below the model card's 21.26. This reruns the model's own recipe (4 beams, 56–142 tokens) on the first 1000 test articles **of any length**, not only those under the worker's 10,000-character limit. ROUGE-2 near 0.19–0.21 means the gap was the short-article subset and sample size; still ~0.15 points at how we score. Takes ~15 min; doesn't affect anything below.

In [ ]:
!python quick_tests/beams_experiment.py --beams 4 --min-length 56 --max-length 142 --all-lengths --num-docs 1000 --out {RESULTS}/beams_experiment_published_check.csv

## 2. Batch-size sweep (model only)

In [ ]:
!python quick_tests/batch_size_sweep.py --beams {BEAM_COUNT} --out {RESULTS}/batch_size_sweep.csv

## 3. End-to-end concurrency sweeps

Each run starts a worker in the background, waits for `/healthz`, runs the sweep (1, 8, 32 and 128 clients, 256 Documents per level) and stops the worker, so only one model is on the GPU at a time.

In [ ]:
import os, subprocess, time, requests

def worker_commit(checkout):
    return subprocess.check_output(["git", "-C", checkout, "rev-parse", "--short", "HEAD"], text=True).strip()

def start_worker(checkout, label, **settings):
    env = {**os.environ, "NUM_BEAMS": str(BEAM_COUNT), "BATCH_WAIT_MS": "50", **{k: str(v) for k, v in settings.items()}}
    log = open(f"/content/worker_{label}.log", "w")
    process = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                               cwd=f"{checkout}/server/worker", env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        if process.poll() is not None:
            raise RuntimeError(f"worker exited, see /content/worker_{label}.log")
        try:
            if requests.get("http://127.0.0.1:8000/healthz", timeout=1).status_code == 200:
                return process
        except requests.ConnectionError:
            pass
        time.sleep(1)
    process.terminate()
    raise TimeoutError(f"worker not ready, see /content/worker_{label}.log")

def stop_worker(process):
    process.terminate()
    process.wait(timeout=60)

def sweep(checkout, label, **settings):
    process = start_worker(checkout, label, **settings)
    try:
        commit = worker_commit(checkout)
        get_ipython().system(f"python quick_tests/concurrency_sweep.py --url http://127.0.0.1:8000/summarize "
                             f"--label {label} --worker-commit {commit} --out {RESULTS}/concurrency_sweep_{label}.csv")
    finally:
        stop_worker(process)

In [ ]:
sweep("/content/gist-baseline", "baseline")

In [ ]:
sweep("/content/gist", "bs1", SUMMARIZATION_BATCH_SIZE=1)

In [ ]:
sweep("/content/gist", "bs8", SUMMARIZATION_BATCH_SIZE=8)

In [ ]:
sweep("/content/gist", "bs32", SUMMARIZATION_BATCH_SIZE=32)

## Results

In [ ]:
import glob
import pandas as pd

beams = pd.read_csv(f"{RESULTS}/beams_experiment.csv")
display(beams.groupby("num_beams")[["latency_s", "rouge1", "rouge2", "rougeLsum"]].mean().round(4))

batch = pd.read_csv(f"{RESULTS}/batch_size_sweep.csv")
ok = batch[batch.status == "ok"]
display(ok.groupby("batch_size").apply(lambda g: pd.Series({"docs_per_s": g.num_docs.sum() / g.latency_s.sum(),
                                                            "batch_latency_s": g.latency_s.mean()})).round(3))
print("Out of memory at batch sizes:", batch[batch.status == "oom"].batch_size.tolist())

e2e = pd.concat(pd.read_csv(path) for path in glob.glob(f"{RESULTS}/concurrency_sweep_*.csv"))
def summarize_level(g):
    ok = g[g.status_code == 200]
    elapsed = (g.sent_at_s + g.latency_s).max()
    return pd.Series({"ok": len(ok), "errors": len(g) - len(ok), "docs_per_s": len(ok) / elapsed,
                      "p50_s": ok.latency_s.quantile(0.5), "p95_s": ok.latency_s.quantile(0.95)})
display(e2e.groupby(["label", "clients"]).apply(summarize_level).round(3))

## Download

Downloads every CSV as one zip. Unzip into `quick_tests/results/` locally.

In [ ]:
!cd {RESULTS} && zip -q -j /content/gist_gpu_results.zip *.csv
from google.colab import files
files.download("/content/gist_gpu_results.zip")